In [1]:
# Cell 1: Setup
"""
# ArcticDEM Strip Difference Analysis
Compute and visualize elevation differences between pairs of DEM strips.
Uses pre-computed zarr files for efficient processing.
"""

import sys
sys.path.append('..')

from config import OUTPUT_DIR
from diff_analysis import run_dem_difference, compute_dem_difference

print("✓ Modules loaded")


Elevation from DEMs Configuration
✓ ARCHIVE_DIR: /home/moralpom/luna/CPOM/archive/SATS/OPTICAL/ArcticDEM/strips/s2s041/2m/
✓ MOSAIC_DIR: /home/moralpom/luna/CPOM/archive/SATS/OPTICAL/ArcticDEM/mosaic/v4.1/2m/
✓ MOSAIC_INDEX_DIR: /home/moralpom/luna/CPOM/archive/SATS/OPTICAL/ArcticDEM/ArcticDEM_Mosaic_Index_latest_shp/
✓ OUTPUT_DIR: /home/moralpom/luna/CPOM/moralpom/globe/data/ArcticDEM/

✓ Modules loaded


In [ ]:
# Cell 2: Configuration
"""
## Configuration
Edit these parameters for your analysis.
"""

# Tile identifier
TILE = "31_38_1_1"  # <-- EDIT THIS

# Directory containing pre-computed zarr files
ZARR_DIR = f"/path/to/filled_arrays/{TILE}/zarr/"  # <-- EDIT THIS
ZARR_DIR = f"/home/moralpom/luna/CPOM/moralpom/globe/data/ArcticDEM/temp/filled_arrays/{TILE}/zarr/"

# Output directory (defaults to config OUTPUT_DIR)
OUTPUT_DIR = None  # None = use config default

# Extent of the tile in EPSG:3413 (for proper georeferencing)
EXTENT = (-300000, -1000000, -250000, -950000)  # <-- EDIT THIS (left, bottom, right, top)

# Optional shapefile for overlay (e.g., lake outlines)
SHP_PATH = None  # <-- EDIT THIS or set to None

# Define strip pairs to compare (recent, old)
# Format: ("newer_strip", "older_strip")
STRIP_PAIRS = [
    (
        "SETSM_s2s041_W2W2_20210420_10300100BD2ED700_10300100BE98CB00_2m_lsf_seg1_dem_mosaic_v_0-0_dh_0-0000_['vertical_offset_mean', 'nuthkaab', 'deramp']_coregistered",
        "SETSM_s2s041_WV02_20220428_10300100D12D1100_10300100D2137900_2m_lsf_seg1_dem_mosaic_v_0-0_dh_0-0000_['vertical_offset_mean', 'nuthkaab', 'deramp']_coregistered"
    ),
    # Add more pairs as needed:
    # ("strip_a_2", "strip_b_2"),
]

print(f"Tile: {TILE}")
print(f"Zarr directory: {ZARR_DIR}")
print(f"Number of pairs: {len(STRIP_PAIRS)}")
print(f"Extent: {EXTENT}")
print(f"Shapefile: {SHP_PATH}")

Tile: 31_38_1_1
Zarr directory: /home/moralpom/luna/CPOM/moralpom/globe/data/ArcticDEM/temp/filled_arrays/31_38_1_1/zarr/
Number of pairs: 1
Extent: (-300000, -1000000, -250000, -950000)
Shapefile: None


In [3]:
# Cell 3: Run Analysis
"""
## Run DEM Difference Analysis
Process all strip pairs and generate difference maps.
"""

results = run_dem_difference(
    tile=TILE,
    strip_pairs=STRIP_PAIRS,
    zarr_dir=ZARR_DIR,
    output_dir=OUTPUT_DIR,
    extent=EXTENT,
    shp_path=SHP_PATH,
    cmap="RdBu_r",
    auto_crop=True,
    crop_threshold=0.5,
)

print("\n✓ Analysis complete!")


DEM Difference Analysis - Tile: 31_38_1_1
Number of pairs: 1
Zarr directory: /home/moralpom/luna/CPOM/moralpom/globe/data/ArcticDEM/temp/filled_arrays/31_38_1_1/zarr/
Processing: SETSM_s2s041_W2W2_20210420_10300100BD2ED... - SETSM_s2s041_WV02_20220428_10300100D12D1...
  Could not find zarr files for SETSM_s2s041_W2W2_20210420_103 or SETSM_s2s041_WV02_20220428_103
  Failed to process pair

Analysis complete. 0 pairs processed.

✓ Analysis complete!


In [4]:
# Cell 4: View Results
"""
## Results Summary
"""

from IPython.display import Image, display

for (strip_a, strip_b), data in results.items():
    print(f"\n{'='*60}")
    print(f"Pair: {strip_a[:40]}...")
    print(f"      {strip_b[:40]}...")
    print(f"{'='*60}")
    
    stats = data['stats']
    print(f"Valid pixels: {stats['valid_pixels']:,}")
    print(f"Mean difference: {stats['mean']:.2f} m")
    print(f"Std difference: {stats['std']:.2f} m")
    print(f"Min/Max: {stats['min']:.2f} / {stats['max']:.2f} m")
    
    if data['plot_path']:
        print(f"\nPlot: {data['plot_path']}")
        display(Image(filename=data['plot_path']))

In [5]:
# Cell 5: Quick Single-Pair Analysis
"""
## Quick Single-Pair Analysis
For testing or quick looks at individual pairs.
"""

# Select a single pair to analyze
test_strip_a = STRIP_PAIRS[0][0]
test_strip_b = STRIP_PAIRS[0][1]

diff, mask = compute_dem_difference(test_strip_a, test_strip_b, ZARR_DIR)

if diff is not None:
    print(f"Valid pixels: {np.sum(mask):,} / {mask.size:,}")
    print(f"Mean difference: {np.mean(diff.compressed()):.2f} m")
    print(f"Std difference: {np.std(diff.compressed()):.2f} m")
    
    # Quick plot
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 10))
    im = ax.imshow(diff, cmap='RdBu_r', vmin=-10, vmax=10)
    plt.colorbar(im, ax=ax, label='Elevation Difference (m)')
    ax.set_title(f"{test_strip_a[:40]}...\n− {test_strip_b[:40]}...")
    plt.tight_layout()
    plt.show()

  Could not find zarr files for SETSM_s2s041_W2W2_20210420_103 or SETSM_s2s041_WV02_20220428_103
